In [39]:
# !rm -rf logs/ # clear logs
# !rm -rf optimizer_output/

# Imports

In [40]:
# # The following workaround is needed to run the jupyter notebook in docker container.
import os
import subprocess

# Run the script in a shell, capture its environment
PDK = "ihp-sg13g2"
command = f"bash -c 'source /foss/tools/sak/iic-pdk-script.sh {PDK} && source ~/.bashrc && env'"
result = subprocess.run(command, capture_output=True, text=True, shell=True)

# Parse environment variables from the output
for line in result.stdout.splitlines():
    key, _, value = line.partition("=")
    if key and value:
        os.environ[key] = value
        
os.environ["PATH"] += ":/foss/tools/bin"

# Now they are in your current Python process
print("PDK_ROOT:", os.environ.get("PDK_ROOT"))
print("SPICE_USERINIT_DIR:", os.environ.get("SPICE_USERINIT_DIR"))

# Test ngspice again
!ngspice -v

PDK_ROOT: /foss/pdks
SPICE_USERINIT_DIR: /foss/pdks/ihp-sg13g2/libs.tech/ngspice
******
** ngspice-44.2 : Circuit level simulation program
** Compiled with KLU Direct Linear Solver
** The U. C. Berkeley CAD Group
** Copyright 1985-1994, Regents of the University of California.
** Copyright 2001-2024, The ngspice team.
** Please get your ngspice manual from https://ngspice.sourceforge.io/docs.html
** Please file your bug-reports at http://ngspice.sourceforge.net/bugrep.html
** Creation Date: Sat May 24 09:38:33 UTC 2025
******


In [41]:
import sympy as sp
import logging

from pathlib import Path

from symxplorer.spice_engine             import Spicelib_Wrapper, Sim_Execution_Type
from symxplorer.designer_tools           import Nevergrad_Spice_Multi_Spec_Optimizer, Project_Setup

from symxplorer.logging import setup_loggers

logger = logging.getLogger("SymXplorer.jupyter")
logger.info("Spicelib_Wrapper imported successfully.")

09:00:06 - SymXplorer.jupyter: [INFO] Spicelib_Wrapper imported successfully.


# Instantiations


In [42]:
# ----------------------------
# Instantiations
# ----------------------------
project_setup_yaml = Path(f"/foss/designs/eda/SymXplorer/examples/tunable-tia/ihp-sg13g2/spice/project_setup.yaml")
_ = setup_loggers()

09:00:06 - SymXplorer: [INFO] 🚀 Logger initialized and ready!
09:00:06 - SymXplorer: [INFO] 📄 Log file: /foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/logs/SymXplorer_2025-10-01_09-00-06.log
09:00:06 - SymXplorer: [INFO] 🔧 spicelib logger set to 50


In [43]:
# (1) Load the project setup information
PROJECT_SETUP = Project_Setup.from_yaml(project_setup_yaml)
PROJECT_SETUP

09:00:07 - SymXplorer.domains: [INFO] 📂 Loading project setup from /foss/designs/eda/SymXplorer/examples/tunable-tia/ihp-sg13g2/spice/project_setup.yaml
09:00:08 - SymXplorer.domains: [INFO] Initialized OptimizerConfig: RandomSearch, type=nevergrad, budget=200, random_seed=48
09:00:08 - SymXplorer.domains: [INFO] 	Linear bounds: min=0, max=100
09:00:08 - SymXplorer.domains: [INFO] 	Log bounds: min=1, max=100
09:00:08 - SymXplorer.domains: [INFO] 	Loss function: max_loss=inf, norm_method=min-max, type=mse, rescale_mag=True, include_phase_loss=False, include_mag_loss=True
09:00:08 - SymXplorer.domains: [INFO] 	Number of target specs: 2
09:00:08 - SymXplorer.domains: [INFO] 		- TargetSpec(name=fc, target=10e6, tolerance=100000.0, goal=OptimizationGoalType.EXACT, sim_type=SimType.AC, enable=True)
09:00:08 - SymXplorer.domains: [INFO] 		- TargetSpec(name=gain_db, target=10, tolerance=1, goal=OptimizationGoalType.EXCEED, sim_type=SimType.AC, enable=True)
09:00:08 - SymXplorer.domains: [INFO]

Project_Setup(name='Tunable-TIA', description='Tunable TIA BPF example sizing in the ihp-sg13g2 technology', simulator='ngspice', ws_root=PosixPath('/foss/designs/eda/SymXplorer'), netlist=PosixPath('examples/tunable-tia/ihp-sg13g2/spice/tb_ac.spice'), outdir=PosixPath('examples/tunable-tia/scripts/optimizer_output'), tech_spec=TechSpec(name='ihp-sg13g2', constraints={'max_nfet_w': np.float64(9.999999999999999e-06), 'min_nfet_w': np.float64(1.8e-07), 'max_nfet_l': np.float64(9.999999999999999e-06), 'min_nfet_l': np.float64(1.8e-07), 'max_pfet_w': np.float64(9.999999999999999e-06), 'min_pfet_w': np.float64(1.8e-07), 'max_pfet_l': np.float64(9.999999999999999e-06), 'min_pfet_l': np.float64(1.8e-07), 'max_cap_w': np.float64(0.01), 'min_cap_w': np.float64(1e-06), 'max_cap_l': np.float64(0.01), 'min_cap_l': np.float64(1e-06), 'max_res_w': np.float64(0.001), 'min_res_w': np.float64(1e-06), 'max_res_l': np.float64(0.001), 'min_res_l': np.float64(1e-06)}), pvt=PVT(temp=25, corner='tt', supply=

In [44]:
# (2) Create the Spice Simulator Wrapper
wrapper = Spicelib_Wrapper(
    project_name=PROJECT_SETUP.name,
    netlist_filename= PROJECT_SETUP.ws_root / PROJECT_SETUP.netlist,
    output_folder=PROJECT_SETUP.ws_root / PROJECT_SETUP.outdir,
    sim_execution_t=Sim_Execution_Type.RUN_AND_WAIT,  # only RUN_AND_WAIT is supported as of now...,
    path_to_simulator=Path("/foss/tools/bin/ngspice"),
    verbose=False
    )
wrapper

09:00:08 - SymXplorer.spicelib: [WARNING] ⚠️ Output directory already exists, re-creating: /foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/optimizer_output
09:00:10 - SymXplorer.spicelib: [INFO] --------------------------------------------------
09:00:10 - SymXplorer.spicelib: [INFO] 🚀 Spicelib_Wrapper initialized successfully!
09:00:10 - SymXplorer.spicelib: [INFO] 	📝 Project: Tunable-TIA
09:00:10 - SymXplorer.spicelib: [INFO] 	📜 Schematic: tb_ac
09:00:10 - SymXplorer.spicelib: [INFO] 	📂 Output Folder: /foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/optimizer_output
09:00:10 - SymXplorer.spicelib: [INFO] --------------------------------------------------
09:00:10 - SymXplorer.spicelib: [INFO] Using ngspice from ['/foss/tools/bin/ngspice']
09:00:10 - SymXplorer.spicelib: [INFO] 📊 --- Circuit Information ---
09:00:10 - SymXplorer.spicelib: [INFO] 🔗 Nodes in the netlist: ['VSS', 'GND', 'VDD', 'Vbias', 'Von', 'Vop', 'In', 'Ip']
09:00:10 - SymXplorer.spicelib: [INFO] Te

In [45]:
circuit_optimizer = Nevergrad_Spice_Multi_Spec_Optimizer(
    spicelib_wrapper=wrapper,
    setup_obj=PROJECT_SETUP
)
circuit_optimizer

09:00:11 - SymXplorer.optimizer: [INFO] Initialized the Nevergrad_Spice_Multi_Spec_Optimizer with 2 target specs


## Sanity Check

In [46]:
wrapper.run_sanity_check(
    use_editor=True,
    sim_execution_t=Sim_Execution_Type.RUN_NOW
)

09:00:11 - SymXplorer.spicelib: [INFO] 📂 Creating dedicated sanity check folder...
09:00:11 - SymXplorer.spicelib: [INFO] 🧪 Running sanity check simulation...


09:00:14 - SymXplorer.spicelib: [INFO] simulator log: /foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/optimizer_output/sanity_check/Tunable-TIA_sanity.log
09:00:14 - SymXplorer.spicelib: [INFO] simulator RAW: /foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/optimizer_output/sanity_check/Tunable-TIA_sanity.raw
09:00:14 - SymXplorer.spicelib: [INFO] 🔎 Verifying simulation results...
09:00:14 - SymXplorer.spicelib: [INFO] ✅ Sanity check passed 🎉


True

# Method Calls

In [47]:
circuit_optimizer.parameterize()

Dict(vbias=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_cap_l=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_cap_w=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_nfet_l=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_nfet_w=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_res_3_l=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_res_3_w=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_res_s_l=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_res_s_w=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}]):{'x_dut_nfet_w': 50.0, 'x_dut_nfet_l': 50.0, 'x_dut_cap_w': 50.0, 'x_dut_cap_l': 50.0, 'x_dut_res_s_l': 50.0, 'x_dut_res_s_w': 50.0, 'x_dut_res_3_l': 50.0, 'x_dut_res_3_w': 50.0, 'vbias': 50.0}

In [48]:
circuit_optimizer.optimize()

09:00:14 - SymXplorer.optimizer: [INFO] Optimization process started.
09:00:14 - SymXplorer.optimizer: [INFO] Optimizer is set to RandomSearch with budget = 200
Optimizing:  24%|██▎       | 47/200 [01:11<03:54,  1.53s/trial]


KeyboardInterrupt: 

In [49]:
circuit_optimizer.plot_loss(save_path=project_setup_yaml.parent / "loss_curve.html", show=True)

09:01:31 - SymXplorer.optimizer: [INFO] 📊 Plot saved to /foss/designs/eda/SymXplorer/examples/tunable-tia/ihp-sg13g2/spice/loss_curve.html
09:01:32 - SymXplorer.optimizer: [INFO] Opening interactive plot in browser...


In [50]:
out = circuit_optimizer.get_best_params()
if out is not None: 
    best_param, loss, metadata = out
metadata

09:01:49 - SymXplorer.optimizer: [INFO] best loss: 6.747115037192609


{'fc': {'curr_val': np.float64(28440470.0),
  'loss': np.float64(3.400509338209)},
 'gain_db': {'curr_val': np.float64(4.215014521207846),
  'loss': np.float64(6.747115037192609)}}

In [ ]:
# Print parameter sizes (convert to u)
for param in best_param:
    print(f"{param}: {best_param[param]*1e6 :0.2f}")

In [ ]:
circuit_optimizer.plot_solution(best_param, show_plot=True, trace_name="vout")

08:58:05 - SymXplorer.optimizer: [INFO] total loss: 10.062022737715125
08:58:05 - SymXplorer.optimizer: [INFO] 	Spec 'fc': curr_val=18641245.0, loss=0.7467111515002499
08:58:05 - SymXplorer.optimizer: [INFO] 	Spec 'gain_db': curr_val=0.34841381626062556, loss=10.062022737715125


In [51]:
circuit_optimizer.plot_optimization_trace(metric_x='fc', metric_y='gain_db', show=True)

09:01:56 - SymXplorer.optimizer: [INFO] Opening interactive plot in browser...


(tensor([29260145.0000, 33347245.0000,  6149836.5000,  7286902.5000,
         21932880.0000, 37929515.0000, 13341805.0000,  4811669.5000,
         38268255.0000,  8199322.0000,  6013164.0000, 43996890.0000,
          5993396.5000,  9176768.5000, 11967540.0000, 20444425.0000,
         17633955.0000,  6262023.0000, 16869850.0000, 19373485.0000,
          5324751.5000, 28440470.0000, 16984535.0000, 11160250.0000,
         15083660.0000, 33092130.0000, 15622775.0000,  5443178.0000,
         12666915.0000, 16485380.0000,  7104205.5000, 24411865.0000,
          7052073.0000,  9254342.5000, 10070930.0000, 20121260.0000,
          6172538.0000,  7382458.0000,  5424878.0000,  6305008.0000,
          8653642.0000,  4476901.0000, 42515515.0000,  6916650.0000,
          9866123.5000,  9352192.5000,  4279039.0000]),
 tensor([ -1.2521,   2.7703, -60.2489, -46.3985, -22.1598,   2.7570, -10.4053,
         -32.2369,   0.6196, -43.4775, -21.0470, -10.5817, -30.6376, -16.0743,
         -23.2303,  -4.6796

# Testing

In [ ]:
circuit_optimizer.optimizer_log

In [ ]:
PROJECT_SETUP.optimizer_config.target_specs.list_target_names()

In [ ]:
target_spec = PROJECT_SETUP.optimizer_config.target_specs.get_target_by_name('gain_db')
target_spec

In [ ]:
circuit_optimizer.compute_spec_loss(spec_curr_val=-90, target_spec=target_spec)

In [ ]:
PROJECT_SETUP.dut_params